# FutureScope: Diagnostic-First Time-Series Forecaster
This notebook demonstrates the capabilities of the fixed and optimized FutureScopeForecaster, focusing on performance, residuals diagnostics, and novel extensions.

In [ ]:
import pandas as pd
import numpy as np
from future_scope_fixed import FutureScopeForecaster
import matplotlib.pyplot as plt
import seaborn as sns

# Sample Data Generation (Seasonal + Noise)
np.random.seed(42)
t = np.arange(1000)
y = 100 + 0.1 * t + 20 * np.sin(2 * np.pi * t / 24) + np.random.normal(0, 5, 1000)
df = pd.DataFrame({'ds': pd.date_range('2023-01-01', periods=1000, freq='H'), 'y': y})

df.head()

## 1. Initializing and Preprocessing

In [ ]:
fs = FutureScopeForecaster(target_col='y', datetime_col='ds', seasonal_period=24)
fs.ingest_data(df)
fs.preprocess(light_mode=True)
print(f"Data shape after preprocessing: {fs.data.shape}")

## 2. Model Selection (Ensemble Mode)

In [ ]:
fs.select_model(mode='ensemble')
print(f"Best Model Order: {fs.best_model['order']}")

## 3. Forecasting with Diagnostics

In [ ]:
forecast = fs.forecast(horizon=48)
fig = fs.diagnostics()
fig.show()

## 4. Novel Extensions
### STL-Transformer Hybrid

In [ ]:
fs.fit_hybrid_transformer(epochs=10)
hybrid_fc = fs.forecast_hybrid(horizon=24)

plt.figure(figsize=(10, 5))
plt.plot(fs.data['y'].tail(100).values, label='Actual')
plt.plot(np.arange(100, 124), hybrid_fc['mean'].values, label='Hybrid Forecast', linestyle='--')
plt.legend()
plt.title("STL-Transformer Hybrid Forecast")
plt.show()

### Bayesian GP Uncertainty

In [ ]:
fs.fit_gp_uncertainty()
gp_fc = fs.forecast_with_gp(horizon=24)

plt.figure(figsize=(10, 5))
plt.plot(gp_fc['mean'].values, label='Mean Forecast')
plt.fill_between(range(len(gp_fc)), gp_fc['mean_ci_lower'], gp_fc['mean_ci_upper'], alpha=0.2, label='95% GP CI')
plt.legend()
plt.title("Forecasting with Bayesian GP Uncertainty")
plt.show()